# CODE 

In [ ]:
import os
import pandas as pd
import numpy as np
import subprocess
from tqdm import tqdm
import warnings
from scipy.spatial.distance import cdist
from pathlib import Path



# 1. CHARGEMENT

print("Chargement des données...")
df_en = pd.read_csv("../../interspeech-2020-perceptimatic/DATA/english/all_aligned_clean_english.csv", sep=r"\s+") 
df_fr = pd.read_csv("../../interspeech-2020-perceptimatic/DATA/french/all_aligned_clean_french.csv", sep=r"\s+")
df_en['language'] = 'english'
df_fr['language'] = 'french'
df_items = pd.concat([df_en, df_fr])

df_tests = pd.read_csv("../../interspeech-2020-perceptimatic/DATA/human_and_models.csv", low_memory=False)
df_triplets = df_tests.drop_duplicates(subset=['filename', 'language']).copy()


# 2. LE MOTEUR KALDI 


def extraire_mfcc_kaldi(filepath):


    """
    extracted with Kaldi toolkit, using the default parameters, 
    adding the first and second derivatives for a total of 39 dimensions, 
    and we apply mean-variance normalization over a moving 300 milliseconds window
    """


    cmd = (
        f"compute-mfcc-feats --snip-edges=true 'scp:echo my_id {filepath} |' ark:- 2>/dev/null | "
        f"add-deltas ark:- ark:- 2>/dev/null | "
        f"apply-cmvn-sliding --cmn-window=300 --center=true ark:- 'ark,t:-' 2>/dev/null"
    )

    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)

    if result.returncode != 0 or not result.stdout.strip():
        return np.empty((0, 39))
    
    lignes = result.stdout.strip().split('\n')
    matrice = []
    for ligne in lignes:
        l = ligne.strip()
        if 'my_id' in l or '[' in l: continue
        if ']' in l: l = l.replace(']', '')
        vals = l.split()
        if vals:
            matrice.append([float(x) for x in vals])
    return np.array(matrice)



def decouper_matrice_kaldi_timestamps(matrice_mfcc, onset, offset):

    """Logique : découpe temporelle t0=0.0, comme dans le script pris de script_get_file_distance.py l18 """
    frames = []
    previous_time = 0.0
    begin = False
    
    for i in range(matrice_mfcc.shape[0]):
        current_time = i * 0.010 + 0.0125 # 10ms par frame + 12.5ms de décalage pour le centre de la fenêtre
        if not begin:
            if onset > previous_time and onset <= current_time:
                begin = True
                frames.append(i)
        else:
            if not (offset >= previous_time and offset < current_time):
                frames.append(i)
            else:
                break
        previous_time = current_time
        
    if not frames:
        return np.empty((0, matrice_mfcc.shape[1]))
    return matrice_mfcc[frames, :]

def get_file_info(item_index, lang):
    row = df_items[(df_items['index'] == item_index) & (df_items['language'] == lang)].iloc[0]
    return f"../../data-wav/{lang}/1s/{row['#file']}.wav", row['onset'], row['offset']


# 3. LE DTW 


def accelerated_dtw_millet(x, y, dist='cosine'):
    """Pris et simplifier de accelerated_dtw() de dtw.py"""
    r, c = len(x), len(y)
    D0 = np.zeros((r + 1, c + 1))
    D0[0, 1:] = np.inf
    D0[1:, 0] = np.inf
    D1 = D0[1:, 1:]
    A = cdist(x, y, dist)
    D0[1:, 1:] = A.copy()
    C = D1.copy()
    for i in range(r):
        for j in range(c):
            min_list = [D0[i, j]]
            for k in range(1, 2):  # warp=1
                min_list += [D0[min(i+k, r-1), j], D0[i, min(j+k, c-1)]]
            D1[i, j] += min(min_list)
    return D1[-1, -1] / float(max(x.shape[0], y.shape[0])) 




def calculer_delta_machine(mfcc_X, mfcc_TGT, mfcc_OTH):
    if mfcc_X.shape[0] < 2 or mfcc_TGT.shape[0] < 2 or mfcc_OTH.shape[0] < 2:
        return np.nan
    try:
        norm_TGT = accelerated_dtw_millet(mfcc_TGT, mfcc_X)
        norm_OTH = accelerated_dtw_millet(mfcc_OTH, mfcc_X)
        return (norm_OTH - norm_TGT)
    except:
        return np.nan

# DOSSIERS DE SAUVEGARDE 
SAVE_DIR = Path().resolve() / "matrices_mfcc_sauvegardees"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

(SAVE_DIR / "english").mkdir(exist_ok=True)
(SAVE_DIR / "french").mkdir(exist_ok=True)

# Dictionnaire pour tracker les statistiques 
stats_matrices = {'disque': 0, 'kaldi': 0}

def load_matrice(filepath, lang, item_id, onset, offset):
    """
    Vérifie si la matrice découpée existe déjà sur le disque.
    Si OUI, la charge instantanément.
    Si NON, demande à Kaldi de l'extraire, la découpe, puis la sauvegarde.
    """
    # On crée un nom de fichier unique basé sur le mot et la langue
    nom_fichier = f"{item_id}.npy"
    chemin_sauvegarde = SAVE_DIR / lang / nom_fichier

    # 1. Si le fichier existe déjà, on le lit 
    if chemin_sauvegarde.exists():
        stats_matrices['disque'] += 1
        return np.load(chemin_sauvegarde)

    # 2. S'il n'existe pas, on lance Kaldi
    stats_matrices['kaldi'] += 1
    matrice_brute = extraire_mfcc_kaldi(filepath)
    matrice_decoupee = decouper_matrice_kaldi_timestamps(matrice_brute, onset, offset)
    

    np.save(chemin_sauvegarde, matrice_decoupee)
    
    return matrice_decoupee


# 4. BOUCLE PRINCIPALE 

deltas_calcules = []

print("Lancement du calcul...")
for index, row in tqdm(df_triplets.iterrows(), total=len(df_triplets), desc="Progression"):
    try:
        lang = row['language']
        path_TGT, on_TGT, off_TGT = get_file_info(row['TGT_item'], lang)
        path_OTH, on_OTH, off_OTH = get_file_info(row['OTH_item'], lang)
        path_X, on_X, off_X = get_file_info(row['X_item'], lang)
        
        mfcc_TGT = load_matrice(path_TGT, lang, row['TGT_item'], on_TGT, off_TGT)
        mfcc_OTH = load_matrice(path_OTH, lang, row['OTH_item'], on_OTH, off_OTH)
        mfcc_X   = load_matrice(path_X, lang, row['X_item'], on_X, off_X)
        
        delta = calculer_delta_machine(mfcc_X, mfcc_TGT, mfcc_OTH)
        
        deltas_calcules.append({
            'filename': row['filename'],
            'language': lang,
            'Delta_Machine_DTW_Kaldi': delta
        })
    except Exception as e:
        print(f"\n🚨 ARRÊT D'URGENCE !")
        print(f"Fichier qui a fait planter : {row['filename']} ({lang})")
        print(f"Cible cherchée : TGT_item = {row['TGT_item']}")
        print(f"Type de l'erreur : {type(e).__name__}")
        print(f"Message exact : {e}")
        break # On casse la boucle pour que tu puisses lire


# 5. SAUVEGARDE ET CORRÉLATION

df_resultats_machine = pd.DataFrame(deltas_calcules)
df_final = pd.merge(df_tests, df_resultats_machine, on=['filename', 'language'], how='left')

# Calcul de la corrélation 
correlation = df_final['Delta_Machine_DTW_Kaldi'].corr(df_final['MFCC'])

# Sauvegarde en fichier CSV
chemin_csv = SAVE_DIR / "resultats_comparatifs_kaldi_dtw.csv"
df_final.to_csv(chemin_csv, index=False)

print(f"\nTerminé !")
print(f"Statistiques : {stats_matrices['disque']} matrices chargées depuis le disque, {stats_matrices['kaldi']} calculées via Kaldi.")
print(f"Corrélation finale : {correlation:.4f}")
print(f"Résultats enregistrés dans : {chemin_csv}")

Chargement des données...
Lancement du calcul...


Progression: 100%|██████████| 5202/5202 [00:12<00:00, 418.75it/s]



Terminé !
Statistiques : 15606 matrices chargées depuis le disque, 0 calculées via Kaldi.
Corrélation finale : 0.9992
Résultats enregistrés dans : /Users/leandreraeth/Desktop/STAGE2026/code/semaine3/matrices_mfcc_sauvegardees/resultats_comparatifs_kaldi_dtw.csv


# ANALYSE CORRELATION 

In [59]:
# Charge le meilleur CSV
df = pd.read_csv("matrices_mfcc_sauvegardees/resultats_comparatifs_kaldi_dtw.csv")
df['Diff_Absolue'] = (df['Delta_Machine_DTW_Kaldi'] - df['MFCC']).abs()
df_propre = df.dropna(subset=['Diff_Absolue'])

# Regardons combien de lignes sont identiques
nb_parfaits = (df_propre['Diff_Absolue'] < 1e-4).sum()
pourcentage_parfait = (nb_parfaits / len(df_propre)) * 100

print(f"Lignes identiques à 10^-4 : {pourcentage_parfait:.2f}% ({nb_parfaits}/{len(df_propre)})")
print(f"Erreur maximale observée : {df_propre['Diff_Absolue'].max():.6f}")

print("\nLES PIRES OUTLIERS")
print(df_propre.sort_values(by='Diff_Absolue', ascending=False)[['filename', 'Delta_Machine_DTW_Kaldi', 'MFCC', 'Diff_Absolue']].head(5))

Lignes identiques à 10^-4 : 1.07% (183/17121)
Erreur maximale observée : 0.035821

LES PIRES OUTLIERS
      filename  Delta_Machine_DTW_Kaldi      MFCC  Diff_Absolue
16439     EN70                 0.127254  0.163075      0.035821
11136     EN69                 0.127254  0.163075      0.035821
10436     EN69                 0.127254  0.163075      0.035821
10521     EN69                 0.127254  0.163075      0.035821
14304     EN70                 0.127254  0.163075      0.035821
